# NASA CMAPSS FD001 — EDA & Feature Engineering Notebook
## Mission Readiness & Predictive Maintenance Hackathon

**Dataset:** Turbofan Engine Degradation Simulation (FD001)
- 100 training engines, 100 test engines
- Single operating condition, single fault mode: HPC Degradation
- 26 columns: unit_number, time_cycles, 3 op settings, 21 sensors

**Goal:** Build a clean, ML-ready dataset for RUL prediction + mission go/no-go logic.

---
**Run the script version instead:** `python cmapss_pipeline.py`

This notebook imports and calls each function from `cmapss_pipeline.py`
so you can inspect outputs cell-by-cell during demos.

In [ ]:
# ─── Cell 0: Imports & Setup ──────────────────────────────────────────────
# All heavy lifting is in cmapss_pipeline.py; this notebook calls those
# functions one by one so judges can see results interactively.

import sys, os
import warnings
warnings.filterwarnings('ignore')

# Make sure the pipeline module is importable from this notebook's location
NOTEBOOK_DIR = os.path.dirname(os.path.abspath('__file__'))
sys.path.insert(0, NOTEBOOK_DIR)

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')   # use 'inline' if running interactively in Jupyter
import matplotlib.pyplot as plt
import seaborn as sns

# Import all pipeline functions
from cmapss_pipeline import (
    load_data,
    analyze_cycle_structure,
    clean_and_smooth,
    extract_critical_sensors,
    compute_rul,
    apply_mission_window,
    normalize_and_export,
    write_eda_summary,
    COLUMNS, TOP_SENSORS, SENSORS_TO_DROP, RUL_CLIP,
    PLOTS_DIR, PROC_DIR
)

print('All imports successful!')
print(f'Plots will be saved to: {PLOTS_DIR}')
print(f'CSVs will be saved to:  {PROC_DIR}')

---
## Step 1 — Load Data
Parse raw space-separated text files, assign column names, run sanity checks.

In [ ]:
# ─── Cell 1: Load ─────────────────────────────────────────────────────────
train_df, test_df, rul_df = load_data()

print('\nColumn names assigned:')
print(train_df.columns.tolist())
print('\nFirst 3 rows of training data:')
train_df.head(3)

---
## Step 2 — Engine Lifespan & Degradation Curves
Understand how long engines live and whether sensors visibly degrade.

In [ ]:
# ─── Cell 2: Cycle Structure ───────────────────────────────────────────────
life_df = analyze_cycle_structure(train_df, test_df)
print('\nEngine lifespan DataFrame (first 5):')
life_df.head()

In [ ]:
# ─── Cell 2b: Display saved plots inline ──────────────────────────────────
from IPython.display import Image, display
import os

for plot in ['01_engine_lifespan_distribution.png', '02_sensor_degradation_curves.png']:
    path = os.path.join(PLOTS_DIR, plot)
    if os.path.exists(path):
        display(Image(filename=path, width=900))
    else:
        print(f'Plot not found: {path}')

---
## Step 3 — Clean & Smooth
Identify useless constant sensors and apply rolling-window smoothing.

In [ ]:
# ─── Cell 3: Clean & Smooth ────────────────────────────────────────────────
train_df, test_df, low_var_sensors = clean_and_smooth(train_df, test_df, window=5)

print(f'\nLow-variance sensors identified: {low_var_sensors}')
print(f'\nNew *_smooth columns added. Train shape: {train_df.shape}')
# Show a sample: raw vs smoothed for sensor_2
sample = train_df[train_df['unit_number']==1][['time_cycles','sensor_2','sensor_2_smooth']].head(10)
print('\nRaw vs Smoothed — unit 1, sensor_2:')
sample

In [ ]:
# Display variance plot inline
from IPython.display import Image, display
display(Image(filename=os.path.join(PLOTS_DIR, '03_sensor_variance.png'), width=900))

---
## Step 4 — Critical Sensor Extraction
Rank all sensors by correlation with RUL. Keep the top 14 most informative.

In [ ]:
# ─── Cell 4: Feature Ranking ───────────────────────────────────────────────
sensor_rank = extract_critical_sensors(train_df, life_df)

print('\nTop 14 sensors by |correlation with RUL|:')
sensor_rank.nlargest(14, 'abs_corr_rul')[['sensor','corr_with_rul','corr_with_cycle']]

In [ ]:
# Display correlation plots inline
from IPython.display import Image, display
for plot in ['04_sensor_correlation_heatmap.png', '05_sensor_rul_correlation_bars.png']:
    display(Image(filename=os.path.join(PLOTS_DIR, plot), width=900))

---
## Step 5 — RUL Computation
Training: ground-truth RUL from data. Test: back-calculated from RUL_FD001.txt.
Clip at 125 to focus model on the degradation window.

In [ ]:
# ─── Cell 5: RUL ──────────────────────────────────────────────────────────
train_df, test_df = compute_rul(train_df, test_df, rul_df, life_df)

print('\nTrain RUL sample:')
train_df[['unit_number','time_cycles','rul','rul_clipped']].head(10)

In [ ]:
from IPython.display import Image, display
display(Image(filename=os.path.join(PLOTS_DIR, '06_rul_distribution.png'), width=900))

---
## Step 6 — Mission Window Logic
Core business logic: Can this engine survive the next mission?
**Change `MISSION_DURATION` below to test different fleet scenarios.**

In [ ]:
# ─── Cell 6: Mission Window ────────────────────────────────────────────────
# Try 30, 45, 60 — or any value your backend config specifies per asset
MISSION_DURATION = 30.0

print(f'Applying mission window logic for mission_duration = {MISSION_DURATION} cycles...')

train_df = apply_mission_window(train_df, MISSION_DURATION)
test_df  = apply_mission_window(test_df,  MISSION_DURATION)

# Show critical risk examples from test set
print('\nSample critical-risk rows from test set:')
risk_rows = test_df[test_df['CRITICAL_OVERLAP_RISK']].tail(10)
risk_rows[['unit_number','time_cycles','rul_clipped',
           'predicted_failure_TTE','mission_duration','margin_of_safety','CRITICAL_OVERLAP_RISK']]

In [ ]:
# ─── Cell 6b: Mission Readiness Quick Visualization ───────────────────────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Plot last-cycle RUL vs mission duration for all test engines
last_cycle_test = test_df.groupby('unit_number').last().reset_index()

fig, ax = plt.subplots(figsize=(14, 5))
colors = ['#DD4444' if r else '#55A868' for r in last_cycle_test['CRITICAL_OVERLAP_RISK']]
ax.bar(last_cycle_test['unit_number'], last_cycle_test['rul_clipped'],
       color=colors, edgecolor='white', alpha=0.85)
ax.axhline(MISSION_DURATION, color='black', linestyle='--', linewidth=2,
           label=f'Mission Duration = {MISSION_DURATION} cycles')
ax.set_xlabel('Engine Unit Number')
ax.set_ylabel('Predicted RUL at Last Known Cycle')
ax.set_title('Test Fleet Mission Readiness (Red = CRITICAL OVERLAP RISK)')
ax.legend()
plt.tight_layout()

plot_path = os.path.join(PLOTS_DIR, '08_mission_readiness_fleet.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.close()
print(f'Plot saved: {plot_path}')

from IPython.display import Image, display
display(Image(filename=plot_path, width=900))

---
## Step 7 — Normalize & Export
Fit StandardScaler on training data only. Export CSVs + schema.json + scaler.pkl.

In [ ]:
# ─── Cell 7: Export ────────────────────────────────────────────────────────
train_final, test_final = normalize_and_export(train_df, test_df, sensor_rank)

print('\nFinal training dataset shape:', train_final.shape)
print('Final test dataset shape:    ', test_final.shape)
print('\nFinal column list:')
print(train_final.columns.tolist())

print('\nSample training rows (scaled):')
train_final.head(5)

In [ ]:
# Verify schema.json was written correctly
import json
schema_path = os.path.join(PROC_DIR, 'schema.json')
with open(schema_path) as f:
    schema = json.load(f)

print(f'Schema has {len(schema["columns"])} columns')
print('\nFirst 5 schema entries:')
for col in schema['columns'][:5]:
    print(f"  {col['name']:30s} | {col['pg_type']:20s} | {col['description']}")

---
## Step 8 — EDA Summary Report

In [ ]:
# ─── Cell 8: Write Summary ─────────────────────────────────────────────────
summary_path = write_eda_summary(
    train_df, test_df, life_df, sensor_rank,
    low_var_sensors, train_final, test_final
)
print(f'\nEDA Summary written to: {summary_path}')

# Print summary inline
with open(summary_path, 'r', encoding='utf-8') as f:
    from IPython.display import Markdown, display
    display(Markdown(f.read()))

---
## Bonus: Try Different Mission Durations
Reusable mission window function — just change the number below!

In [ ]:
# ─── Cell 9: Mission Sensitivity Analysis ─────────────────────────────────
# How does fleet risk change as mission_duration increases?

durations = [10, 20, 30, 40, 50, 60, 75, 100]
risk_pcts = []

for dur in durations:
    tmp = apply_mission_window(test_df, dur)
    last = tmp.groupby('unit_number')['CRITICAL_OVERLAP_RISK'].last()
    risk_pcts.append(100 * last.sum() / len(last))

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(durations, risk_pcts, marker='o', color='#DD4444', linewidth=2, markersize=8)
ax.fill_between(durations, risk_pcts, alpha=0.15, color='#DD4444')
ax.set_xlabel('Mission Duration (cycles)')
ax.set_ylabel('% Fleet at CRITICAL OVERLAP RISK')
ax.set_title('Fleet Risk vs Mission Duration — Sensitivity Analysis')
ax.set_ylim(0, 105)
ax.grid(True, alpha=0.3)
plt.tight_layout()

plot_path = os.path.join(PLOTS_DIR, '09_mission_duration_sensitivity.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.close()

from IPython.display import Image, display
display(Image(filename=plot_path, width=800))

print('Mission Duration -> Risk %:')
for d, r in zip(durations, risk_pcts):
    print(f'  {d:3d} cycles: {r:.1f}% at risk')

---
## Pipeline Complete!

**Outputs generated:**
- `processed/train_ready.csv` — ML training set
- `processed/test_ready.csv` — ML test set  
- `processed/schema.json` — PostgreSQL-ready column schema
- `processed/scaler.pkl` — Fitted StandardScaler for inference
- `processed/plots/` — 9 diagnostic plots
- `EDA_SUMMARY.md` — Human-readable summary for the team

**Next steps for teammates:**
- **ML Team:** Use `train_ready.csv` with LSTM/GRU/XGBoost for RUL regression
- **Backend Team:** Use `schema.json` to design PostgreSQL table structures
- **API Team:** Import `apply_mission_window()` directly from `cmapss_pipeline.py`